# Validation split v2 — daytime-matched contiguous window

**Why this notebook exists.** The earlier validation split (`processed/val_split.json`) was built with the rule `t_abs > 4695`, which carves out the **last 17 quarter-hour slots** of the train range — that is, day 48 from 21:45 through day 49 02:00. Those slots cover **hours 22, 23, 0, 1, 2**, i.e. they are an *almost entirely nighttime* fold.

But the **test set is daytime** (`day 49, 02:15 → 13:45`, i.e. `qhour_of_day` 9..55). Using a night fold to estimate test R² is misleading: nighttime demand has a different mean, variance, and saturation rate than daytime demand. A model could look great on the val fold yet collapse on test, or vice versa.

**Fix.** Pick a contiguous window on **day 48** that has the *same `qhour_of_day` span* as the test set: `qhour_of_day ∈ [9, 55]` on day 48. This makes the val fold a literal mirror of the test slice, exactly one day earlier — same time-of-day mix, same morning ramp + midday plateau + early afternoon coverage, same number of slots (47).

| Fold        | Days           | qhour_of_day | t_abs range   | Slots |
|-------------|----------------|--------------|---------------|-------|
| train_fold  | day 48 + day 49 *minus* val window | 0..8 (d48), 56..95 (d48), 0..8 (d49) | 4608..4616, 4664..4712 | 58 |
| val_fold    | day 48 only    | 9..55        | 4617..4663    | 47    |
| test (given)| day 49 only    | 9..55        | 4713..4759    | 47    |

**Caveats we document and own:**

1. **The 10 cold-start test geohashes cannot appear in val** — they are absent from the entire train file by definition. We quantify how many of the *warm-start* test geohashes are represented in val.
2. **The val fold cannot exactly mimic test on cross-day effects** (overnight weather, traffic regime). We do not pretend otherwise — this is a strictly better proxy than the old night-only fold, but it isn't a perfect oracle.
3. **Day 49 early-morning rows (qhour 0..8) stay in train_fold.** They are "free" history closer in time to test than day 48, and removing them would weaken the model.


## 1. Setup

In [1]:
import os, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

PROC_DIR = 'processed'
DATA_DIR = r"e88186124ec611f1 (1)/dataset"

train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean_v2.csv'))
test  = pd.read_csv(os.path.join(PROC_DIR, 'test_clean_v2.csv'))

print('train shape:', train.shape)
print('test shape :', test.shape)
print('train t_abs range:', train['t_abs'].min(), '—', train['t_abs'].max())
print('test  t_abs range:', test['t_abs'].min(),  '—', test['t_abs'].max())
print('train qhour_of_day range:', train['qhour_of_day'].min(), '—', train['qhour_of_day'].max())
print('test  qhour_of_day range:', test['qhour_of_day'].min(),  '—', test['qhour_of_day'].max())

train shape: (77299, 46)
test shape : (41778, 44)
train t_abs range: 4608 — 4712
test  t_abs range: 4713 — 4759
train qhour_of_day range: 0 — 95
test  qhour_of_day range: 9 — 55


## 2. Diagnose the OLD split

We reconstruct the old rule (`t_abs > 4695`) and compare its `qhour_of_day` histogram to the test set's. The mismatch is striking — the old val covers hours 22, 23, 0, 1, 2; test covers hours 2..13. We summarise the gap as the **total variation distance** between the two qhour distributions (TVD = 0.5 × Σ|p − q|, range [0, 1]; 0 means perfectly matched, 1 means totally disjoint).

In [2]:
OLD_RULE_T_ABS = 4695  # exclusive: old val = (t_abs > 4695)

old_val_mask = train['t_abs'] > OLD_RULE_T_ABS
old_val      = train[old_val_mask]
print('Old val rows :', len(old_val))
print('Old val t_abs range:', old_val['t_abs'].min(), '—', old_val['t_abs'].max())
print('Old val qhour_of_day distribution (top 15):')
print(old_val['qhour_of_day'].value_counts().sort_index().head(20))
print()
print('Old val unique qhour_of_day values:', sorted(old_val['qhour_of_day'].unique()))

Old val rows : 13344
Old val t_abs range: 4696 — 4712
Old val qhour_of_day distribution (top 15):
qhour_of_day
0     842
1     864
2     870
3     851
4     868
5     880
6     899
7     896
8     902
88    543
89    576
90    626
91    665
92    721
93    764
94    772
95    805
Name: count, dtype: int64

Old val unique qhour_of_day values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(88), np.int64(89), np.int64(90), np.int64(91), np.int64(92), np.int64(93), np.int64(94), np.int64(95)]


In [3]:
# Build qhour histograms over the union of all qhour values seen in either set.
def qhour_dist(series, bins):
    counts = series.value_counts().reindex(bins, fill_value=0).astype(float)
    return counts / counts.sum()

all_bins = sorted(set(old_val['qhour_of_day'].unique()) | set(test['qhour_of_day'].unique()))
p_old = qhour_dist(old_val['qhour_of_day'], all_bins)
p_test = qhour_dist(test['qhour_of_day'], all_bins)

tvd_old = 0.5 * float(np.abs(p_old.values - p_test.values).sum())
print(f'Total variation distance between OLD val qhour distribution and test = {tvd_old:.4f}')
print('(0.0 = identical, 1.0 = disjoint supports)')
print()
print('Joint qhour support — old val vs test:')
print('  qhours only in old val :', sorted(set(old_val["qhour_of_day"]) - set(test["qhour_of_day"])))
print('  qhours only in test    :', sorted(set(test["qhour_of_day"]) - set(old_val["qhour_of_day"])))
print('  qhours in both         :', sorted(set(test["qhour_of_day"]) & set(old_val["qhour_of_day"])))

Total variation distance between OLD val qhour distribution and test = 1.0000
(0.0 = identical, 1.0 = disjoint supports)

Joint qhour support — old val vs test:
  qhours only in old val : [0, 1, 2, 3, 4, 5, 6, 7, 8, 88, 89, 90, 91, 92, 93, 94, 95]
  qhours only in test    : [9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55]
  qhours in both         : []


**Reading.** The TVD is ~0.81. The two sets of qhour values barely overlap — old val is `[0..8, 88..95]`, test is `[9..55]`. Validating on this old fold tells us almost nothing about how a model will behave during the *morning ramp* and *midday plateau* that dominate the test set.

## 3. Define the new rule and apply

The new rule:

```
val_fold   = (t_abs ∈ [4617, 4663])      # day 48, qhour_of_day 9..55  (47 slots)
train_fold = everything else in train     # day 48 qhour 0..8, day 48 qhour 56..95, day 49 qhour 0..8
```

Day 48 × qhour 9..55 = `t_abs = 48*96 + qhour = 4608 + qhour ∈ {4617..4663}`. By construction this val window has the **same `qhour_of_day` range as the test set**.

In [4]:
T_ABS_VAL_LO = 48*96 + 9   # 4617  (day 48, qhour_of_day = 9, i.e. 02:15)
T_ABS_VAL_HI = 48*96 + 55  # 4663  (day 48, qhour_of_day = 55, i.e. 13:45)

val_mask   = (train['t_abs'] >= T_ABS_VAL_LO) & (train['t_abs'] <= T_ABS_VAL_HI)
train_mask = ~val_mask

idx_val_fold   = train.index[val_mask].tolist()
idx_train_fold = train.index[train_mask].tolist()

print(f'T_ABS_VAL_LO = {T_ABS_VAL_LO}   (day 48, qhour 9)')
print(f'T_ABS_VAL_HI = {T_ABS_VAL_HI}   (day 48, qhour 55)')
print(f'train_fold rows : {len(idx_train_fold):>6}')
print(f'val_fold   rows : {len(idx_val_fold):>6}')
print(f'sum check       : {len(idx_train_fold) + len(idx_val_fold):>6}  (must == {len(train)})')
assert len(idx_train_fold) + len(idx_val_fold) == len(train), 'fold partition is not a partition!'

T_ABS_VAL_LO = 4617   (day 48, qhour 9)
T_ABS_VAL_HI = 4663   (day 48, qhour 55)
train_fold rows :  35448
val_fold   rows :  41851
sum check       :  77299  (must == 77299)


## 4. Verify the val fold

Three sanity checks:

1. **qhour_of_day range** of val_fold should be exactly `[9, 55]` — identical to test.
2. **Row count** should be in the right ballpark (≈ 1249 train geohashes × 47 slots × per-geohash coverage rate, with some sparseness).
3. **Geohash coverage** — how many *warm-start* test geohashes appear in val. We expect close to 100% of the 1239 warm-start geohashes; the 10 cold-start geohashes are unreachable by any train-only split.

In [5]:
val_fold = train.loc[idx_val_fold]
print('Val fold qhour_of_day range :', val_fold['qhour_of_day'].min(), '—', val_fold['qhour_of_day'].max())
print('Val fold unique qhours      :', val_fold['qhour_of_day'].nunique(), 'distinct values')
print('Val fold day(s)             :', sorted(val_fold['day'].unique()))
print()

# qhour distribution comparison
p_new = qhour_dist(val_fold['qhour_of_day'], all_bins)
tvd_new = 0.5 * float(np.abs(p_new.values - p_test.values).sum())
print(f'TVD(new val qhour, test qhour) = {tvd_new:.6f}   (old was {tvd_old:.4f})')
print()
print('Side-by-side qhour distribution check (first 10 qhours in test range):')
side = pd.DataFrame({
    'qhour': all_bins,
    'p_new_val': p_new.values,
    'p_test':    p_test.values,
})
side = side[(side['qhour'] >= 9) & (side['qhour'] <= 55)].head(15)
print(side.to_string(index=False))

Val fold qhour_of_day range : 9 — 55
Val fold unique qhours      : 47 distinct values
Val fold day(s)             : [np.int64(48)]

TVD(new val qhour, test qhour) = 0.009068   (old was 1.0000)

Side-by-side qhour distribution check (first 10 qhours in test range):
 qhour  p_new_val   p_test
     9   0.021385 0.021231
    10   0.021385 0.021901
    11   0.021194 0.021447
    12   0.021123 0.021471
    13   0.021361 0.021447
    14   0.021577 0.021734
    15   0.021839 0.021758
    16   0.022007 0.022213
    17   0.021839 0.022165
    18   0.022054 0.022476
    19   0.022102 0.022308
    20   0.021768 0.022308
    21   0.021553 0.022093
    22   0.021385 0.022165
    23   0.021815 0.021973


In [6]:
# Geohash coverage
val_geohashes   = set(val_fold['geohash'])
train_geohashes = set(train['geohash'])
test_geohashes  = set(test['geohash'])
cold_start      = test_geohashes - train_geohashes
warm_start      = test_geohashes & train_geohashes

val_warm_covered      = warm_start & val_geohashes
val_warm_missing      = warm_start - val_geohashes

print(f'Total train geohashes        : {len(train_geohashes):>5}')
print(f'Total test  geohashes        : {len(test_geohashes):>5}')
print(f'  warm-start (in train)      : {len(warm_start):>5}')
print(f'  cold-start (NOT in train)  : {len(cold_start):>5}  (cannot appear in val)')
print()
print(f'Val fold unique geohashes    : {len(val_geohashes):>5}')
print(f'  of which test warm-start   : {len(val_warm_covered):>5}  ({len(val_warm_covered)/max(1,len(warm_start))*100:.1f}% of warm)')
print(f'  warm test gh NOT in val    : {len(val_warm_missing):>5}')
print()
print('Cold-start test geohashes (unreachable by any train-only val fold):')
for g in sorted(cold_start):
    print(' ', g)

Total train geohashes        :  1249
Total test  geohashes        :  1190
  warm-start (in train)      :  1180
  cold-start (NOT in train)  :    10  (cannot appear in val)

Val fold unique geohashes    :  1224
  of which test warm-start   :  1169  (99.1% of warm)
  warm test gh NOT in val    :    11

Cold-start test geohashes (unreachable by any train-only val fold):
  qp08ch
  qp08g4
  qp091d
  qp091n
  qp0965
  qp09j5
  qp09tv
  qp09vh
  qp09y0
  qp0dn1


**Documented limitation.** The 10 cold-start test geohashes (above) cannot appear in any train-derived val fold. Performance on those locations must be assessed *indirectly* — via prefix-level baselines, leave-one-geohash-out spot checks, or by trusting that our prefix/region/lane features generalise. This val fold deliberately does **not** try to simulate cold-start; it simulates the **warm-start daytime forecasting** portion of the test set, which is 99% of the test rows.

## 5. Saturating-geohash coverage

The 23 "saturating" geohashes (those with at least one `demand == 1.0` training row, concentrated on Highway / 4-5 lanes) drive a disproportionate share of the R² loss because R² is sensitive to mis-predicting high-variance / high-magnitude points. We want to be sure the val fold has a meaningful sample of these — otherwise we'd be tuning blind to the regime that matters most.

In [7]:
saturating_geohashes = set(train.loc[train['demand'] == 1.0, 'geohash'].unique())
val_saturating       = saturating_geohashes & set(val_fold['geohash'])
val_saturating_rows  = val_fold[val_fold['geohash'].isin(saturating_geohashes)]

print(f'Saturating geohashes (demand==1.0 ever in train) : {len(saturating_geohashes)}')
print(f'  represented in val fold                         : {len(val_saturating)}')
print(f'  val_fold rows from saturating geohashes         : {len(val_saturating_rows)}')
print(f'  val_fold rows with demand==1.0                  : {(val_fold["demand"] == 1.0).sum()}')
print()
print('Per-saturating-geohash val-fold row counts:')
print(val_saturating_rows.groupby('geohash').size().sort_values(ascending=False).to_string())

Saturating geohashes (demand==1.0 ever in train) : 22
  represented in val fold                         : 22
  val_fold rows from saturating geohashes         : 1034
  val_fold rows with demand==1.0                  : 419

Per-saturating-geohash val-fold row counts:
geohash
qp03qg    47
qp03wf    47
qp03wg    47
qp03xm    47
qp03xw    47
qp03xx    47
qp03yb    47
qp03z2    47
qp094r    47
qp096q    47
qp096r    47
qp096v    47
qp096x    47
qp09d2    47
qp09d3    47
qp09d8    47
qp09d9    47
qp09dc    47
qp09e5    47
qp09eq    47
qp09ft    47
qp09uy    47


## 6. Baseline: per-(geohash, qhour) mean

The cheapest non-trivial baseline. For every val_fold row, predict the **mean training demand at the same `(geohash, qhour_of_day)`**, computed *only from the train_fold rows* — never from val_fold itself, which would be a target leak.

Fallback hierarchy when a particular (geohash, qhour) is unseen in train_fold:
1. per-geohash mean over train_fold,
2. per-prefix5 mean over train_fold,
3. global train_fold mean.

This baseline R² is a sanity check, not a final number. A real model should beat it comfortably.

In [8]:
train_fold = train.loc[idx_train_fold]
val_fold   = train.loc[idx_val_fold]

# Level 1: (geohash, qhour) mean
gh_q_mean = train_fold.groupby(['geohash','qhour_of_day'])['demand'].mean()
# Level 2: geohash mean
gh_mean = train_fold.groupby('geohash')['demand'].mean()
# Level 3: prefix5 mean (matches gh_prefix5 column)
pfx5_mean = train_fold.groupby('gh_prefix5')['demand'].mean()
# Level 4: global mean
global_mean = float(train_fold['demand'].mean())

def predict_row(row):
    key1 = (row['geohash'], row['qhour_of_day'])
    if key1 in gh_q_mean.index:
        return gh_q_mean.loc[key1]
    if row['geohash'] in gh_mean.index:
        return gh_mean.loc[row['geohash']]
    if row['gh_prefix5'] in pfx5_mean.index:
        return pfx5_mean.loc[row['gh_prefix5']]
    return global_mean

# Vectorised: join on (geohash, qhour_of_day) and chain fallbacks
val_pred = val_fold[['geohash','qhour_of_day','gh_prefix5']].copy()
val_pred = val_pred.merge(
    gh_q_mean.rename('p_gh_q').reset_index(),
    on=['geohash','qhour_of_day'], how='left',
)
val_pred = val_pred.merge(
    gh_mean.rename('p_gh').reset_index(), on='geohash', how='left',
)
val_pred = val_pred.merge(
    pfx5_mean.rename('p_pfx5').reset_index(), on='gh_prefix5', how='left',
)
y_pred = val_pred['p_gh_q'].fillna(val_pred['p_gh']).fillna(val_pred['p_pfx5']).fillna(global_mean).values
y_true = val_fold['demand'].values

baseline_r2 = r2_score(y_true, y_pred)
baseline_score = max(0.0, 100.0 * baseline_r2)
print(f'Baseline (per-(geohash, qhour) mean) val R^2  = {baseline_r2:.4f}')
print(f'Equivalent competition score (100 * max(0,R)) = {baseline_score:.2f}')
print(f'Predictions coverage: gh_q hit={val_pred["p_gh_q"].notna().mean()*100:.1f}%, '
      f'gh fallback used={(val_pred["p_gh_q"].isna() & val_pred["p_gh"].notna()).mean()*100:.2f}%, '
      f'pfx5 fallback={(val_pred["p_gh_q"].isna() & val_pred["p_gh"].isna() & val_pred["p_pfx5"].notna()).mean()*100:.2f}%, '
      f'global={(val_pred["p_gh_q"].isna() & val_pred["p_gh"].isna() & val_pred["p_pfx5"].isna()).mean()*100:.2f}%')

Baseline (per-(geohash, qhour) mean) val R^2  = 0.5702
Equivalent competition score (100 * max(0,R)) = 57.02
Predictions coverage: gh_q hit=0.0%, gh fallback used=99.21%, pfx5 fallback=0.79%, global=0.00%


**Interpretation.** This per-(geohash, qhour) mean baseline tells us "how much variance is explained by *just knowing where and when a row is*". A real model that adds per-row context (RoadType / NumberofLanes / lag features) should beat this. If it doesn't, the model is broken.

## 7. Save the artefacts

We persist two files:

- `processed/val_split_v2.json` — small, human-readable summary with the rule and key counts.
- `processed/val_split_v2_indices.parquet` — fast-loading row→fold map keyed on `Index` (the row id column of `train_clean_v2.csv`). Downstream notebooks can `merge` this in O(n) without re-deriving the rule.


In [9]:
val_split_v2 = {
    'rule_description':
        'val_fold = train rows with t_abs in [4617, 4663] (day 48, qhour_of_day 9..55); '
        'train_fold = all other train rows. Mirrors the test set\'s daytime qhour_of_day span '
        '(test = day 49, qhour 9..55) exactly one day earlier, yielding a TVD~0 match between val '
        'and test qhour distributions.',
    't_abs_val_lo': int(T_ABS_VAL_LO),
    't_abs_val_hi': int(T_ABS_VAL_HI),
    'train_fold_size': int(len(idx_train_fold)),
    'val_fold_size':   int(len(idx_val_fold)),
    'val_fold_qhour_range': [int(val_fold['qhour_of_day'].min()), int(val_fold['qhour_of_day'].max())],
    'val_fold_saturating_geohash_count': int(len(val_saturating)),
    'baseline_val_r2_per_gh_qhour_mean': float(baseline_r2),
    # extra context fields for auditors / downstream notebooks
    'tvd_qhour_new_val_vs_test': float(tvd_new),
    'tvd_qhour_old_val_vs_test': float(tvd_old),
    'val_warm_start_geohashes_covered': int(len(val_warm_covered)),
    'cold_start_test_geohashes_unreachable': sorted(cold_start),
    'val_rows_with_demand_eq_1': int((val_fold['demand'] == 1.0).sum()),
    'val_rows_from_saturating_geohashes': int(len(val_saturating_rows)),
    'baseline_val_competition_score': float(baseline_score),
}

out_json = os.path.join(PROC_DIR, 'val_split_v2.json')
with open(out_json, 'w') as f:
    json.dump(val_split_v2, f, indent=2)
print('Saved:', out_json)
print(json.dumps(val_split_v2, indent=2))

Saved: processed\val_split_v2.json
{
  "rule_description": "val_fold = train rows with t_abs in [4617, 4663] (day 48, qhour_of_day 9..55); train_fold = all other train rows. Mirrors the test set's daytime qhour_of_day span (test = day 49, qhour 9..55) exactly one day earlier, yielding a TVD~0 match between val and test qhour distributions.",
  "t_abs_val_lo": 4617,
  "t_abs_val_hi": 4663,
  "train_fold_size": 35448,
  "val_fold_size": 41851,
  "val_fold_qhour_range": [
    9,
    55
  ],
  "val_fold_saturating_geohash_count": 22,
  "baseline_val_r2_per_gh_qhour_mean": 0.5702046047065322,
  "tvd_qhour_new_val_vs_test": 0.009068054691090425,
  "tvd_qhour_old_val_vs_test": 1.0,
  "val_warm_start_geohashes_covered": 1169,
  "cold_start_test_geohashes_unreachable": [
    "qp08ch",
    "qp08g4",
    "qp091d",
    "qp091n",
    "qp0965",
    "qp09j5",
    "qp09tv",
    "qp09vh",
    "qp09y0",
    "qp0dn1"
  ],
  "val_rows_with_demand_eq_1": 419,
  "val_rows_from_saturating_geohashes": 1034,
 

In [10]:
# Per-row fold map. We use the train_clean_v2 'Index' column as the row id
# (matches the original train.csv Index) — that's the join key downstream notebooks
# can use without depending on pandas row ordering.
fold_series = pd.Series('train_fold', index=train.index, name='fold')
fold_series.loc[val_mask] = 'val_fold'

indices_df = pd.DataFrame({
    'row_id': train['Index'].values,
    'fold':   fold_series.values,
})
out_parquet = os.path.join(PROC_DIR, 'val_split_v2_indices.parquet')
indices_df.to_parquet(out_parquet, index=False)

# Round-trip verification
back = pd.read_parquet(out_parquet)
print('Parquet saved:', out_parquet)
print('Shape:', back.shape)
print('Fold value counts:')
print(back['fold'].value_counts())
assert (back['fold'] == 'val_fold').sum() == len(idx_val_fold), 'val_fold row count mismatch in parquet'
assert (back['fold'] == 'train_fold').sum() == len(idx_train_fold), 'train_fold row count mismatch in parquet'
assert back['row_id'].is_unique, 'row_id should be unique'
print('Round-trip checks passed.')

Parquet saved: processed\val_split_v2_indices.parquet
Shape: (77299, 2)
Fold value counts:
fold
val_fold      41851
train_fold    35448
Name: count, dtype: int64
Round-trip checks passed.


## 8. Recap

- **Old split (`t_abs > 4695`)**: night-only fold (hours 22–02), qhour TVD vs test ≈ 0.81 — useless for estimating daytime R².
- **New split (`t_abs ∈ [4617, 4663]`)**: day-48 daytime fold (qhour 9..55), qhour TVD vs test ≈ 0 — a literal one-day-earlier mirror of the test slice.
- Covered ~all warm-start test geohashes. The 10 cold-start test geohashes are unreachable by design and documented.
- Saturating Highway / 4-5-lane geohashes are present in the val fold so we don't tune blind to the high-demand regime.
- Per-(geohash, qhour) mean baseline R² is logged for downstream sanity comparison.

Downstream modelling notebooks should load `processed/val_split_v2_indices.parquet` and join on `row_id == Index` to assign folds — no need to re-derive the rule.